In [10]:
import os
import pandas as pd
from docx import Document
Dir_github = r'X:\PROJECTS\ABC@UofSC\3-Experiments\NSF_valence\github'
project = 'nsf_valence'
pipeline = 'segmentation_Anyssa'
Dir_working = os.path.join(Dir_github,project,pipeline)
Dir_segments = os.path.join(Dir_working,'segmentation')
# Dir_instruct = os.path.join(Dir_working,'instructions')

In [142]:
def parse_segmentation(Dir_segments,file):
    filepath = os.path.join(Dir_segments,file)
    story = file.split('_')[1].split('.')[0]


    doc = Document(filepath)
    lines = []
    for paragraph in doc.paragraphs:
        line = paragraph.text.strip()
        if line:  # Ignore empty lines
            lines.append(line)
    
    df = pd.DataFrame({'segment':lines})
    df['flag'] = 'reason'
    df.loc[(df['segment'].str.startswith('[')) &( df['segment'].str.endswith(']')),'flag'] = 'segment'
    if not df.loc[0,'segment'].startswith('['):
        df.loc[0,'flag'] = 'title'

    df_segment = []
    List_idx = df.loc[df['flag']=='segment'].index
    for i,idx in enumerate(List_idx):
        text = df.loc[idx,'segment'][1:-1]

        if idx != List_idx[-1]:
            List_idx_reason = list(range(idx+1,List_idx[i+1]))
        else:
            List_idx_reason = list(range(idx+1,df.shape[0]))

        if not List_idx_reason:
            reason = '-'
        else:
            reason = '; '.join(df.loc[List_idx_reason,'segment'].values)

            reason = '; '.join(df.loc[List_idx_reason,'segment'].values)
        df_segment.append(pd.DataFrame({'segment':text,'reason':reason},index=[0]))
                
    df_segment = pd.concat(df_segment,ignore_index=True)
    df_segment['story'] = story
    df_segment['idx'] = df_segment.index

    file_output = os.path.join(Dir_segments,f'segment_{story}.csv')
    if not os.path.exists(file_output):
        df_segment.to_csv(file_output,index=False)
    return(df_segment)

In [146]:
List_files = [x for x in os.listdir(Dir_segments) if '.docx' in x]
List_files
dfs_segment = []
for file in List_files:
    
    df_segment = parse_segmentation(Dir_segments,file)
    dfs_segment.append(df_segment)
dfs_segment = pd.concat(dfs_segment,ignore_index=True)
dfs_segment

,segment,reason,story,idx
0,"Long, long ago",positioning element – generic convention,AgedMother,0
1,there lived at the foot of the mountain,positioning element - location,AgedMother,1
2,"a poor farmer and his aged, widowed mother.",>maximal segment,AgedMother,2
3,They owned a bit of land,-,AgedMother,3
4,"which supplied them with food,",relative clause,AgedMother,4
...,...,...,...,...
756,It was a fact of my life,-,Elvis,128
757,that I was not gonna have hair,relative clause,Elvis,129
758,that was anything like Elvis's anytime soon.,relative clause; >maximal segment,Elvis,130
759,I then looked up at the barber with my wet red...,-,Elvis,131
